In [2]:
# load parent annotations
import scanpy as sc
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"

adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated.h5ad")
adata1.obs["cell_type_fine"] = adata1.obs["cell_type"].astype(str)
print(f"Loaded: {adata1.n_obs} cells")
print(adata1.obs["cell_type"].value_counts())

Loaded: 44662 cells
cell_type
T cells                                               20213
CD8/Effector T cells                                   7861
Macrophages                                            5914
NK/Cytotoxic T cells                                   4929
B cells                                                3369
Mixed/stromal-contaminated (CD8+fibroblast signal)     1335
Mast cells                                              494
Monocytes/DC                                            359
pDC                                                     188
Name: count, dtype: int64


In [2]:
# Step 2 — CD4/Treg sub-clustering, rebuilt from the clean parent
# "T cells" cluster ONLY (20,213 cells) — not the old pooled file that
# mixed in CD8/NK cells and caused cross-contamination.

import scanpy.external as sce

t_cells_mask = adata1.obs["cell_type"] == "T cells"
adata1_tcells_raw = adata1.raw.to_adata()[t_cells_mask.values].copy()
adata1_tcells_raw.obs = adata1.obs[t_cells_mask].copy()

print(f"T cells pool: {adata1_tcells_raw.n_obs} cells")

sc.pp.highly_variable_genes(adata1_tcells_raw, n_top_genes=2000, flavor="seurat")
adata1_tcells_hvg = adata1_tcells_raw[:, adata1_tcells_raw.var.highly_variable].copy()
sc.pp.scale(adata1_tcells_hvg, max_value=10)
sc.tl.pca(adata1_tcells_hvg, svd_solver="arpack", random_state=0)
sce.pp.harmony_integrate(adata1_tcells_hvg, key="patient", basis="X_pca", random_state=0)
sc.pp.neighbors(adata1_tcells_hvg, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)

for res in [0.3, 0.5, 0.7]:
    sc.tl.leiden(adata1_tcells_hvg, resolution=res, key_added=f"tcell_leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
    n_clusters = adata1_tcells_hvg.obs[f"tcell_leiden_{res}"].nunique()
    print(f"Resolution {res}: {n_clusters} clusters")

T cells pool: 20213 cells


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-21 15:33:03,962 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-21 15:33:16,033 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-21 15:33:16,390 - harmonypy - INFO - Iteration 1 of 10
2026-07-21 15:33:48,413 - harmonypy - INFO - Iteration 2 of 10
2026-07-21 15:34:14,961 - harmonypy - INFO - Converged after 2 iterations
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resolution 0.3: 4 clusters
Resolution 0.5: 5 clusters
Resolution 0.7: 6 clusters


In [3]:
# Check resolution 0.5 clusters — should now be purely CD4-lineage
# states
adata1_tcells_raw.obs["tcell_leiden_0.5"] = adata1_tcells_hvg.obs["tcell_leiden_0.5"].values

sc.tl.rank_genes_groups(adata1_tcells_raw, groupby="tcell_leiden_0.5", method="wilcoxon")

for cl in sorted(adata1_tcells_raw.obs["tcell_leiden_0.5"].unique(), key=int):
    top_genes = sc.get.rank_genes_groups_df(adata1_tcells_raw, group=cl).head(10)
    n = (adata1_tcells_raw.obs["tcell_leiden_0.5"] == cl).sum()
    print(f"\n=== Cluster {cl} (n={n}) — top genes ===")
    print(top_genes[["names", "logfoldchanges", "pvals_adj"]].to_string(index=False))

# Also check CD4/CD8/NK markers directly, to confirm no CD8/NK
# contamination slipped in via the neighbour graph
print("\n\n=== CD4/CD8/NK marker check per cluster ===")
for gene in ["CD4", "CD8A", "CD8B", "FOXP3", "NKG7", "GNLY"]:
    if gene in adata1_tcells_raw.var_names:
        X = adata1_tcells_raw[:, gene].X
        if hasattr(X, "toarray"): X = X.toarray().flatten()
        adata1_tcells_raw.obs[f"_{gene}"] = X
        means = adata1_tcells_raw.obs.groupby("tcell_leiden_0.5", observed=True)[f"_{gene}"].mean()
        print(f"{gene}: {dict(means.round(2))}")


=== Cluster 0 (n=8532) — top genes ===
  names  logfoldchanges  pvals_adj
  CXCR4        3.271812        0.0
  DUSP1        3.952209        0.0
TSC22D3        2.911392        0.0
   BTG1        1.854929        0.0
    FOS        4.426829        0.0
ZFP36L2        2.001315        0.0
   CD69        3.192381        0.0
   KLF6        2.781787        0.0
   SRGN        2.635481        0.0
   JUNB        2.181225        0.0

=== Cluster 1 (n=2992) — top genes ===
 names  logfoldchanges     pvals_adj
   B2M        0.494675 1.851569e-237
S100A4        1.900409 1.360361e-208
  IL32        1.623460 6.144552e-199
  ACTB        0.889626 2.741684e-184
 HLA-A        0.821046 1.561874e-174
  CD74        1.791107 1.841414e-143
  PFN1        1.047449 8.052863e-119
 HLA-B        0.520064 3.314864e-117
TMSB4X        0.480686 2.141362e-101
 GAPDH        1.201269  4.999506e-99

=== Cluster 2 (n=7555) — top genes ===
names  logfoldchanges  pvals_adj
RPLP2        0.699269        0.0
 RPS6        0.993326 

In [4]:
# CD4 sub-clustering labels — corrected understanding: cluster 3 is
# genuinely CD8 cells that were always misclassified in the parent
# "T cells" cluster cluster 4 is non-T-cell contamination.

tcell_labels = {
    "0": "CD4 Activated T cells",
    "1": "Regulatory T cells (Tregs, CD4+)",
    "2": "CD4 Naive/Resting T cells",
    "3": "CD8 T cells (reclassified from T cells parent cluster)",
    "4": "Non-T-cell contamination (from T cells parent cluster)",
}

adata1_tcells_raw.obs["tcell_subtype_v2"] = adata1_tcells_raw.obs["tcell_leiden_0.5"].map(tcell_labels)

n_unmapped = adata1_tcells_raw.obs["tcell_subtype_v2"].isna().sum()
if n_unmapped > 0:
    raise ValueError(f"{n_unmapped} cells failed to map — check tcell_labels")

print("T cell parent-cluster breakdown, corrected:")
print(adata1_tcells_raw.obs["tcell_subtype_v2"].value_counts())

adata1_tcells_raw.write(PROCESSED_DIR / "GSE114725_tcells_subclustered_v2.h5ad", compression="gzip")
print("\nSaved standalone file — clean, single-source-cluster sub-clustering this time")

T cell parent-cluster breakdown, corrected:
tcell_subtype_v2
CD4 Activated T cells                                     8532
CD4 Naive/Resting T cells                                 7555
Regulatory T cells (Tregs, CD4+)                          2992
CD8 T cells (reclassified from T cells parent cluster)    1090
Non-T-cell contamination (from T cells parent cluster)      44
Name: count, dtype: int64

Saved standalone file — clean, single-source-cluster sub-clustering this time


In [5]:
# Merge CD4 sub-clustering (3 pure states, this time) + macrophage
# sub-clustering into cell_type_fine. The reclassified CD8 (1,090) and
# contamination (44) from the T cells cluster get merged in as their
# own temporary labels for now — CD8 portion will be folded into the
# full CD8 pool in the next step.
adata1.obs["cell_type_fine"] = adata1.obs["cell_type"].astype(str)

adata1.obs.loc[adata1_tcells_raw.obs_names, "cell_type_fine"] = \
    adata1_tcells_raw.obs["tcell_subtype_v2"].astype(str).values

adata1_mac = sc.read_h5ad(PROCESSED_DIR / "GSE114725_macrophages_subclustered.h5ad")
adata1.obs.loc[adata1_mac.obs_names, "cell_type_fine"] = adata1_mac.obs["mac_subtype"].astype(str).values

print("After CD4 (corrected) + macrophage merge:")
print(adata1.obs["cell_type_fine"].value_counts())

After CD4 (corrected) + macrophage merge:
cell_type_fine
CD4 Activated T cells                                     8532
CD8/Effector T cells                                      7861
CD4 Naive/Resting T cells                                 7555
NK/Cytotoxic T cells                                      4929
B cells                                                   3369
Regulatory T cells (Tregs, CD4+)                          2992
Mixed/stromal-contaminated (CD8+fibroblast signal)        1335
Monocyte-like macrophages                                 1265
CD8 T cells (reclassified from T cells parent cluster)    1090
Complement-high macrophages                               1075
LAM-like macrophages                                       946
Antigen-presenting macrophages                             910
Lipid-laden/Foam-cell macrophages                          735
Mast cells                                                 494
Resting/Resident macrophages                               46

In [6]:
# Re-apply NK/cytotoxic 3-way split
adata1_raw = adata1.raw.to_adata()
adata1_raw.obs["cell_type"] = adata1.obs["cell_type"].values

nk_mask = adata1.obs["cell_type"] == "NK/Cytotoxic T cells"
nk_subset_raw = adata1_raw[nk_mask.values].copy()

def get_expr(adata, gene):
    if gene not in adata.var_names:
        return None
    X = adata[:, gene].X
    if hasattr(X, "toarray"):
        X = X.toarray()
    return X.flatten()

cd3d = get_expr(nk_subset_raw, "CD3D")
cd3e = get_expr(nk_subset_raw, "CD3E")
ncam1 = get_expr(nk_subset_raw, "NCAM1")
klrf1 = get_expr(nk_subset_raw, "KLRF1")
fcgr3a = get_expr(nk_subset_raw, "FCGR3A")

t_cell_marker = (cd3d > 0) | (cd3e > 0)
nk_marker = (ncam1 > 0) | (klrf1 > 0) | (fcgr3a > 0)

is_true_nk = ~t_cell_marker & nk_marker
is_nkt = t_cell_marker & nk_marker
is_cytotoxic_cd8 = t_cell_marker & ~nk_marker

nk_barcodes = nk_subset_raw.obs_names
adata1.obs.loc[nk_barcodes[is_true_nk], "cell_type_fine"] = "True NK cells"
adata1.obs.loc[nk_barcodes[is_nkt], "cell_type_fine"] = "NKT cells"
adata1.obs.loc[nk_barcodes[is_cytotoxic_cd8], "cell_type_fine"] = "Cytotoxic CD8 T cells (reclassified)"

print("After NK split re-applied:")
print(adata1.obs["cell_type_fine"].value_counts())

After NK split re-applied:
cell_type_fine
CD4 Activated T cells                                     8532
CD8/Effector T cells                                      7861
CD4 Naive/Resting T cells                                 7555
B cells                                                   3369
Regulatory T cells (Tregs, CD4+)                          2992
True NK cells                                             2192
Mixed/stromal-contaminated (CD8+fibroblast signal)        1335
Monocyte-like macrophages                                 1265
NK/Cytotoxic T cells                                      1146
CD8 T cells (reclassified from T cells parent cluster)    1090
Complement-high macrophages                               1075
LAM-like macrophages                                       946
Antigen-presenting macrophages                             910
Cytotoxic CD8 T cells (reclassified)                       843
NKT cells                                                  748
Lipid-laden/F

In [7]:
#Complete CD8 pool: original CD8/Effector T cells (7,861) +
# reclassified from NK split (843) + reclassified from T cells cluster
# (1,090) = 9,794 cells total
cd8_mask = adata1.obs["cell_type_fine"].isin([
    "CD8/Effector T cells",
    "Cytotoxic CD8 T cells (reclassified)",
    "CD8 T cells (reclassified from T cells parent cluster)"
])
adata1_cd8 = adata1[cd8_mask].copy()
print(f"Complete CD8 pool: {adata1_cd8.n_obs} cells (expected: 9,794)")

adata1_cd8_raw = adata1.raw.to_adata()[cd8_mask.values].copy()
adata1_cd8_raw.obs = adata1_cd8.obs.copy()

sc.pp.highly_variable_genes(adata1_cd8_raw, n_top_genes=2000, flavor="seurat")
adata1_cd8_hvg = adata1_cd8_raw[:, adata1_cd8_raw.var.highly_variable].copy()
sc.pp.scale(adata1_cd8_hvg, max_value=10)
sc.tl.pca(adata1_cd8_hvg, svd_solver="arpack", random_state=0)
sce.pp.harmony_integrate(adata1_cd8_hvg, key="patient", basis="X_pca", random_state=0)
sc.pp.neighbors(adata1_cd8_hvg, use_rep="X_pca_harmony", n_neighbors=15, n_pcs=30, random_state=0)

for res in [0.3, 0.5, 0.7]:
    sc.tl.leiden(adata1_cd8_hvg, resolution=res, key_added=f"cd8_leiden_{res}",
                 flavor="igraph", n_iterations=2, directed=False, random_state=0)
    n_clusters = adata1_cd8_hvg.obs[f"cd8_leiden_{res}"].nunique()
    print(f"Resolution {res}: {n_clusters} clusters")

Complete CD8 pool: 9794 cells (expected: 9,794)


C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
2026-07-21 15:47:57,198 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-07-21 15:48:03,191 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-07-21 15:48:03,407 - harmonypy - INFO - Iteration 1 of 10
2026-07-21 15:48:16,536 - harmonypy - INFO - Iteration 2 of 10
2026-07-21 15:48:25,785 - harmonypy - INFO - Converged after 2 iterations


Resolution 0.3: 4 clusters
Resolution 0.5: 5 clusters
Resolution 0.7: 6 clusters


In [8]:
# Check resolution 0.5 via unbiased DE 
adata1_cd8_raw.obs["cd8_leiden_0.5"] = adata1_cd8_hvg.obs["cd8_leiden_0.5"].values

sc.tl.rank_genes_groups(adata1_cd8_raw, groupby="cd8_leiden_0.5", method="wilcoxon")

for cl in sorted(adata1_cd8_raw.obs["cd8_leiden_0.5"].unique(), key=int):
    top_genes = sc.get.rank_genes_groups_df(adata1_cd8_raw, group=cl).head(12)
    n = (adata1_cd8_raw.obs["cd8_leiden_0.5"] == cl).sum()
    print(f"\n=== Cluster {cl} (n={n}) — top genes ===")
    print(top_genes[["names", "logfoldchanges", "pvals_adj"]].to_string(index=False))


=== Cluster 0 (n=4941) — top genes ===
  names  logfoldchanges     pvals_adj
  CXCR4        2.475841  0.000000e+00
  DUSP1        2.621975  0.000000e+00
   BTG1        1.586577 4.016053e-292
TSC22D3        2.165860 9.618078e-283
  ZFP36        2.264888 5.822181e-252
ZFP36L2        1.451304 6.422361e-206
   KLF6        2.074223 1.016175e-197
   CCL4        2.399507 2.446718e-197
   CCL5        1.740724 1.750346e-188
   RGS1        2.551517 1.242085e-187
   CD69        1.893979 3.870189e-148
    FOS        2.124445 4.138717e-145

=== Cluster 1 (n=1668) — top genes ===
   names  logfoldchanges     pvals_adj
    NKG7        2.674076 6.869673e-180
    FLNA        2.134909 6.066682e-161
    GNLY        3.036167 6.347076e-157
     B2M        0.475527 7.349701e-145
    GZMH        3.315075 4.559177e-134
   HLA-B        0.611319 1.168891e-124
    PFN1        1.370247 1.502212e-115
  FGFBP2        5.966293 1.091056e-103
    PRF1        2.177056 4.681446e-102
    ACTB        0.883192  1.459383e-

In [9]:
# Quick check — does cluster 2 show any secondary GZMK signal, or is
# it genuinely just low-activation/ribosomal-dominant 
for gene in ["GZMK", "GZMA", "IL7R", "CCR7", "TCF7", "SELL"]:
    if gene in adata1_cd8_raw.var_names:
        X = adata1_cd8_raw[:, gene].X
        if hasattr(X, "toarray"): X = X.toarray().flatten()
        adata1_cd8_raw.obs[f"_{gene}"] = X
        means = adata1_cd8_raw.obs.groupby("cd8_leiden_0.5", observed=True)[f"_{gene}"].mean()
        print(f"{gene}: {dict(means.round(2))}")

GZMK: {'0': np.float32(0.85), '1': np.float32(0.67), '2': np.float32(0.52), '3': np.float32(0.7), '4': np.float32(0.85)}
GZMA: {'0': np.float32(0.95), '1': np.float32(1.49), '2': np.float32(0.43), '3': np.float32(1.15), '4': np.float32(1.51)}
IL7R: {'0': np.float32(1.51), '1': np.float32(0.69), '2': np.float32(1.61), '3': np.float32(0.92), '4': np.float32(0.43)}
CCR7: {'0': np.float32(0.08), '1': np.float32(0.05), '2': np.float32(0.59), '3': np.float32(0.16), '4': np.float32(0.2)}
TCF7: {'0': np.float32(0.39), '1': np.float32(0.43), '2': np.float32(1.28), '3': np.float32(0.63), '4': np.float32(0.58)}
SELL: {'0': np.float32(0.1), '1': np.float32(0.27), '2': np.float32(0.91), '3': np.float32(1.16), '4': np.float32(0.51)}


In [10]:
cd8_labels_v2 = {
    "0": "Activated CD8 T cells",
    "1": "Effector CD8 T cells",
    "2": "Naive/Memory CD8 T cells",
    "3": "NK-like CD8 T cells",
    "4": "Cycling CD8 T cells",
}

adata1_cd8_raw.obs["cd8_subtype_v2"] = adata1_cd8_raw.obs["cd8_leiden_0.5"].map(cd8_labels_v2)

n_unmapped = adata1_cd8_raw.obs["cd8_subtype_v2"].isna().sum()
if n_unmapped > 0:
    raise ValueError(f"{n_unmapped} cells failed to map — check cd8_labels_v2")

print("CD8 sub-type counts (complete, corrected pool):")
print(adata1_cd8_raw.obs["cd8_subtype_v2"].value_counts())

adata1_cd8_raw.write(PROCESSED_DIR / "GSE114725_cd8_subclustered_v2.h5ad", compression="gzip")
print("\nSaved standalone file immediately")

CD8 sub-type counts (complete, corrected pool):
cd8_subtype_v2
Activated CD8 T cells       4941
Naive/Memory CD8 T cells    2246
Effector CD8 T cells        1668
NK-like CD8 T cells          840
Cycling CD8 T cells           99
Name: count, dtype: int64

Saved standalone file immediately


In [11]:
adata1.obs["cell_type_fine"] = adata1.obs["cell_type_fine"].astype(str)
adata1.obs.loc[adata1_cd8_raw.obs_names, "cell_type_fine"] = \
    adata1_cd8_raw.obs["cd8_subtype_v2"].astype(str).values

print("Final, complete cell_type_fine:")
print(adata1.obs["cell_type_fine"].value_counts())
print(f"\nTotal: {adata1.obs['cell_type_fine'].value_counts().sum()} (expected: 44,662)")

adata1.write(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad", compression="gzip")
print("\nSaved complete, corrected cell_type_fine for GSE114725")

Final, complete cell_type_fine:
cell_type_fine
CD4 Activated T cells                                     8532
CD4 Naive/Resting T cells                                 7555
Activated CD8 T cells                                     4941
B cells                                                   3369
Regulatory T cells (Tregs, CD4+)                          2992
Naive/Memory CD8 T cells                                  2246
True NK cells                                             2192
Effector CD8 T cells                                      1668
Mixed/stromal-contaminated (CD8+fibroblast signal)        1335
Monocyte-like macrophages                                 1265
NK/Cytotoxic T cells                                      1146
Complement-high macrophages                               1075
LAM-like macrophages                                       946
Antigen-presenting macrophages                             910
NK-like CD8 T cells                                        840
NKT cell

In [12]:
# Inevstigate "Unclear" cells — broader marker panel + raw summed expression
unclear_mask = adata1.obs["cell_type_fine"] == "NK/Cytotoxic T cells"
unclear_raw = adata1.raw.to_adata()[unclear_mask.values].copy()

t_lineage_genes = [g for g in ["CD3D", "CD3E", "CD8A", "CD8B"] if g in unclear_raw.var_names]
nk_lineage_genes = [g for g in ["NCAM1", "KLRF1", "FCGR3A"] if g in unclear_raw.var_names]

def raw_sum(adata, genes):
    X = adata[:, genes].X
    if hasattr(X, "toarray"): X = X.toarray()
    return X.sum(axis=1)

t_score = raw_sum(unclear_raw, t_lineage_genes)
nk_score = raw_sum(unclear_raw, nk_lineage_genes)

is_t_leaning = t_score > nk_score
is_nk_leaning = nk_score > t_score
is_tied_zero = (t_score == 0) & (nk_score == 0)

print(f"T-leaning (reclassify as Cytotoxic CD8): {is_t_leaning.sum()}")
print(f"NK-leaning (reclassify as True NK): {is_nk_leaning.sum()}")
print(f"Genuinely zero on both (no signal at all): {is_tied_zero.sum()}")

T-leaning (reclassify as Cytotoxic CD8): 352
NK-leaning (reclassify as True NK): 0
Genuinely zero on both (no signal at all): 794


In [14]:
import numpy as np

unclear_mask = adata1.obs["cell_type_fine"] == "NK/Cytotoxic T cells"
print(f"Excluding {unclear_mask.sum()} cells from cell_type_fine (kept in cell_type)")

adata1.obs["cell_type_fine"] = adata1.obs["cell_type_fine"].astype(str)
adata1.obs.loc[unclear_mask, "cell_type_fine"] = np.nan

print("\ncell_type_fine, excluded cells removed:")
print(adata1.obs["cell_type_fine"].value_counts(dropna=True))
print(f"\nTotal cells with a fine label: {adata1.obs['cell_type_fine'].notna().sum()}")
print(f"Total cells excluded from fine analysis: {adata1.obs['cell_type_fine'].isna().sum()}")

adata1.write(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad", compression="gzip")
print("\nSaved")

Excluding 1146 cells from cell_type_fine (kept in cell_type)

cell_type_fine, excluded cells removed:
cell_type_fine
CD4 Activated T cells                                     8532
CD4 Naive/Resting T cells                                 7555
Activated CD8 T cells                                     4941
B cells                                                   3369
Regulatory T cells (Tregs, CD4+)                          2992
Naive/Memory CD8 T cells                                  2246
True NK cells                                             2192
Effector CD8 T cells                                      1668
Mixed/stromal-contaminated (CD8+fibroblast signal)        1335
Monocyte-like macrophages                                 1265
Complement-high macrophages                               1075
LAM-like macrophages                                       946
Antigen-presenting macrophages                             910
NK-like CD8 T cells                                        840
N

In [16]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_finelabels"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [17]:
#viability check
import pandas as pd
import numpy as np

MIN_CELLS_PER_SAMPLE = 10
MIN_VIABLE_SAMPLES_PER_GROUP = 3

def check_viability(adata_obs, fine_col, sample_col, group_col, group_values, dataset_name):
    print(f"\n{'='*70}")
    print(f"VIABILITY CHECK — {dataset_name}")
    print(f"{'='*70}")

    results = []
    fine_categories = adata_obs[fine_col].dropna().unique()

    for cat in sorted(fine_categories):
        subset = adata_obs[adata_obs[fine_col] == cat]
        row = {"cell_type_fine": cat, "total_cells": len(subset)}
        all_viable = True
        for group_val in group_values:
            group_subset = subset[subset[group_col] == group_val]
            per_sample_counts = group_subset.groupby(sample_col, observed=True).size()
            n_viable_samples = (per_sample_counts >= MIN_CELLS_PER_SAMPLE).sum()
            row[f"{group_val}_viable_samples"] = n_viable_samples
            if n_viable_samples < MIN_VIABLE_SAMPLES_PER_GROUP:
                all_viable = False
        row["VIABLE_FOR_DE"] = all_viable
        results.append(row)

    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    return df

# ----------------------------
# GSE114725 — Tumour vs Normal
# ----------------------------
viability_114725 = check_viability(
    adata1.obs, "cell_type_fine", "patient", "tissue",
    ["TUMOR", "NORMAL"], "GSE114725 (Tumour vs Normal)"
)
viability_114725.to_csv(RESULTS_DIR / "GSE114725_finelabels_viability_check.csv", index=False)


VIABILITY CHECK — GSE114725 (Tumour vs Normal)
                                        cell_type_fine  total_cells  TUMOR_viable_samples  NORMAL_viable_samples  VIABLE_FOR_DE
                                 Activated CD8 T cells         4941                     8                      4           True
                        Antigen-presenting macrophages          910                     7                      2          False
                                               B cells         3369                     8                      3           True
                                 CD4 Activated T cells         8532                     8                      3           True
                             CD4 Naive/Resting T cells         7555                     7                      2          False
                           Complement-high macrophages         1075                     8                      2          False
                                   Cycling CD8 T cells  

In [18]:
# GSE176078 viability check — three pairwise subtype comparisons,
# same thresholds as GSE114725 (>=10 cells/sample, >=3 viable samples per group).

adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected_finelabels.h5ad")
print(f"Loaded: {adata2.n_obs} cells")

def check_viability_pairwise(adata_obs, fine_col, sample_col, group_col, pairwise_comparisons, dataset_name):
    print(f"\n{'='*70}")
    print(f"VIABILITY CHECK — {dataset_name}")
    print(f"{'='*70}")

    fine_categories = adata_obs[fine_col].dropna().unique()
    results = []

    for cat in sorted(fine_categories):
        subset = adata_obs[adata_obs[fine_col] == cat]
        row = {"cell_type_fine": cat, "total_cells": len(subset)}
        any_viable = False
        for group_a, group_b in pairwise_comparisons:
            comp_name = f"{group_a}_vs_{group_b}"
            n_a = subset[subset[group_col] == group_a].groupby(sample_col, observed=True).size()
            n_b = subset[subset[group_col] == group_b].groupby(sample_col, observed=True).size()
            viable_a = (n_a >= MIN_CELLS_PER_SAMPLE).sum()
            viable_b = (n_b >= MIN_CELLS_PER_SAMPLE).sum()
            comp_viable = (viable_a >= MIN_VIABLE_SAMPLES_PER_GROUP) and (viable_b >= MIN_VIABLE_SAMPLES_PER_GROUP)
            row[f"{comp_name}_viable"] = comp_viable
            if comp_viable:
                any_viable = True
        row["VIABLE_FOR_ANY_COMPARISON"] = any_viable
        results.append(row)

    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    return df

pairwise_comparisons_176078 = [("TNBC", "ER+"), ("HER2+", "ER+"), ("TNBC", "HER2+")]

viability_176078 = check_viability_pairwise(
    adata2.obs, "cell_type_fine", "orig.ident", "subtype",
    pairwise_comparisons_176078, "GSE176078 (pairwise subtypes)"
)
viability_176078.to_csv(RESULTS_DIR / "GSE176078_finelabels_viability_check.csv", index=False)

Loaded: 91425 cells

VIABILITY CHECK — GSE176078 (pairwise subtypes)
                                    cell_type_fine  total_cells  TNBC_vs_ER+_viable  HER2+_vs_ER+_viable  TNBC_vs_HER2+_viable  VIABLE_FOR_ANY_COMPARISON
                                           B cells         2786                True                 True                  True                       True
                                  Basal epithelial         1073               False                False                 False                      False
                                              CAFs         6469                True                 True                  True                       True
                                   Cycling T cells          991               False                False                  True                       True
                                Cycling epithelial         2774                True                 True                  True                       True
       